<center>
<h1><b>Information Retrieval : RAG (Retrieval Augmented generation)</b>
</center>


<center>
<h1><b>BAKKAR Alaaeddine G1_EMSICFC</b>
</center>


## **The Building Blocks of RAG**

### **A Workflow for RAG**

In [1]:
import json
import tiktoken

#import pandas as pd

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader
from langchain_community.vectorstores import Chroma

from dotenv.ipython import load_dotenv
import os

c:\Users\hp\Desktop\WORKDIR\IA_AGENTS\RAGv2\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


# **LLM**

In [2]:
load_dotenv(override=True)

True

In [3]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# **Implementing RAG**

## **1 - Loading the PDF, Chunking**

In [4]:
pdf_file = "./pdfs/Top250movies.pdf"

In [5]:
pdf_loader = PyPDFLoader(pdf_file)

In [6]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
loader = PyPDFDirectoryLoader(path = "./pdfs")

In [7]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='o200k_base',
    chunk_size=300,
    chunk_overlap=20
)

In [8]:
chunks = loader.load_and_split(text_splitter)

In [9]:
len(chunks)

31

## **2 - Vector Store - ChromaDB, Embeddings**

In [10]:
from langchain_openai import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(model='text-embedding-ada-002')

In [11]:
vectorstore = Chroma.from_documents(
    chunks,
    embedding_model,
    collection_name="Movies",
    persist_directory="./store"
)

#### Retrieve Relevant Context

In [12]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [13]:
retrieved_chunks=retriever.invoke("What are the top 10 movies?")

In [14]:
print(retrieved_chunks)

[Document(metadata={'page_label': '1', 'creator': 'PScript5.dll Version 5.2.2', 'author': 'Costas', 'title': 'Microsoft Word - Document1', 'total_pages': 16, 'producer': 'Acrobat Distiller 10.1.7 (Windows)', 'source': 'pdfs\\Top250movies.pdf', 'creationdate': '2014-11-09T00:27:09+02:00', 'moddate': '2014-11-09T00:27:09+02:00', 'page': 0}, page_content="Top 250 \nAs voted by regular IMDb users \nSort by:  \nShowing 250 Titles \n Rank & Title IMDb \nRating \nYour \nRating \n \n1. The Shawshank Redemption (1994)  9.2  RATE  \n \n2. The Godfather (1972)  9.2  RATE  \n \n3. The Godfather: Part II (1974)  9.0  RATE  \n \n4. The Dark Knight (2008)  8.9  RATE  \n \n5. Pulp Fiction (1994)  8.9  RATE  \n \n6. The Good, the Bad and the Ugly (1966)  8.9  RATE  \n \n7. Schindler's List (1993)  8.9  RATE  \n \n8. 12 Angry Men (1957)  8.9  RATE  \n \n9. The Lord of the Rings: The Return of the King \n(2003)  8.9  RATE  \n \n10. Fight Club (1999)  8.8  RATE  \n \n11. The Lord of the Rings: The Fellows

In [15]:
print(len(retrieved_chunks))

5


## **3 - RAG Q&A**

### Generate Answer with LLM

In [16]:
prompt_template = """
Answer the following question based only on provided context
The context is about Top 250  movies
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer : I DON'T KNOW
<context>
 {context}
</context>
<question>
 {question}
</question>
"""

### **Retrieving the Relevant Documents**

In [17]:
user_input = "Title of top 10 movies?"

In [18]:
relevant_document_chunks = retriever.invoke(user_input)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)

In [19]:
context_for_query

"Top 250 \nAs voted by regular IMDb users \nSort by:  \nShowing 250 Titles \n Rank & Title IMDb \nRating \nYour \nRating \n \n1. The Shawshank Redemption (1994)  9.2  RATE  \n \n2. The Godfather (1972)  9.2  RATE  \n \n3. The Godfather: Part II (1974)  9.0  RATE  \n \n4. The Dark Knight (2008)  8.9  RATE  \n \n5. Pulp Fiction (1994)  8.9  RATE  \n \n6. The Good, the Bad and the Ugly (1966)  8.9  RATE  \n \n7. Schindler's List (1993)  8.9  RATE  \n \n8. 12 Angry Men (1957)  8.9  RATE  \n \n9. The Lord of the Rings: The Return of the King \n(2003)  8.9  RATE  \n \n10. Fight Club (1999)  8.8  RATE  \n \n11. The Lord of the Rings: The Fellowship of the Ring \n(2001)  8.8  RATE. Rank & Title IMDb \nRating \nYour \nRating \n \n47. The Departed (2006)  8.5  RATE  \n \n48. Gladiator (2000)  8.5  RATE  \n \n49. Back to the Future (1985)  8.5  RATE  \n \n50. Alien (1979)  8.5  RATE  \n \n51. The Prestige (2006)  8.4  RATE  \n \n52. The Dark Knight Rises (2012)  8.4  RATE  \n \n53. The Lives of O

In [20]:
# Here the length is 10 because, earlier we have declared k=10
len(relevant_document_chunks)

5

In [21]:
prompt = prompt_template.format(context=context_for_query, question=user_input)

In [22]:
print(prompt)


Answer the following question based only on provided context
The context is about Top 250  movies
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer : I DON'T KNOW
<context>
 Top 250 
As voted by regular IMDb users 
Sort by:  
Showing 250 Titles 
 Rank & Title IMDb 
Rating 
Your 
Rating 
 
1. The Shawshank Redemption (1994)  9.2  RATE  
 
2. The Godfather (1972)  9.2  RATE  
 
3. The Godfather: Part II (1974)  9.0  RATE  
 
4. The Dark Knight (2008)  8.9  RATE  
 
5. Pulp Fiction (1994)  8.9  RATE  
 
6. The Good, the Bad and the Ugly (1966)  8.9  RATE  
 
7. Schindler's List (1993)  8.9  RATE  
 
8. 12 Angry Men (1957)  8.9  RATE  
 
9. The Lord of the Rings: The Return of the King 
(2003)  8.9  RATE  
 
10. Fight Club (1999)  8.8  RATE  
 
11. The Lord of the Rings: The Fellowship of the Ring 
(2001)  8.8  RATE. Rank & Title IMDb 
Rating 
Your 
Rating 
 
47. The Departed (2006)  8.5  RATE  
 
4

In [23]:
resp = llm.invoke(prompt)

In [24]:
from IPython.display import Markdown

In [25]:
print(display(Markdown(resp.content)))

1. The Shawshank Redemption (1994)  
2. The Godfather (1972)  
3. The Godfather: Part II (1974)  
4. The Dark Knight (2008)  
5. Pulp Fiction (1994)  
6. The Good, the Bad and the Ugly (1966)  
7. Schindler's List (1993)  
8. 12 Angry Men (1957)  
9. The Lord of the Rings: The Return of the King (2003)  
10. Fight Club (1999)  

None


### **Defining the RAG function for response**




In [26]:
def RAG(query, llm=llm, prompt_template=prompt_template):
    context_docs = retriever.invoke(query)
    context_list = [d.page_content for d in context_docs]
    context_for_query = ". ".join(context_list)
    prompt = prompt_template.format(context=context_for_query, question=query)
    resp=llm.invoke(prompt)
    return resp.content

In [27]:
response = RAG("Rating of the movie The usual suspects")
print(display(Markdown(response)))

The rating of the movie The Usual Suspects is 8.6.

None


In [28]:
user_input = "j'ai fain et je veux manger quelque chose"
output = RAG(user_input)
print(output)

I DON'T KNOW


In [29]:
response = RAG("Director of the movie The usual suspects")
print(display(Markdown(response)))

I DON'T KNOW

None


In [30]:
response = RAG("Rank of the movie The usual suspects")
print(display(Markdown(response)))

24

None


## **4 - Evaluation**

In [31]:
user_input ="Rating and title of top 10 movies"

In [32]:
relevant_document_chunks = retriever.invoke(user_input)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)

In [33]:
user_message_template = """
 ###Question
 {question}
 ###Context
 {context}
 ###Answer
 {answer}
"""

In [34]:
# Default answer for an RAG query
answer = RAG(user_input)
print(display(Markdown(answer)))

1. The Shawshank Redemption (1994) - 9.2  
2. The Godfather (1972) - 9.2  
3. The Godfather: Part II (1974) - 9.0  
4. The Dark Knight (2008) - 8.9  
5. Pulp Fiction (1994) - 8.9  
6. The Good, the Bad and the Ugly (1966) - 8.9  
7. Schindler's List (1993) - 8.9  
8. 12 Angry Men (1957) - 8.9  
9. The Lord of the Rings: The Return of the King (2003) - 8.9  
10. Fight Club (1999) - 8.8  

None


### 1. Groundness

In [35]:
groundedness_rater_system_message="""
Vous êtes chargé d’évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d’IA pour générer la réponse, ainsi qu’une réponse générée par l’IA à la question.

Dans l’entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par l’IA commencera par ###Answer.

Critères d’évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1 — La métrique n’est pas respectée du tout
2 — La métrique n’est respectée que dans une mesure limitée
3 — La métrique est respectée dans une bonne mesure
4 — La métrique est respectée en grande partie
5 — La métrique est entièrement respectée

Métrique :
La réponse doit être dérivée uniquement des informations présentées dans le contexte.

Instructions :

Écrivez d’abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d’évaluation et attribuer un score.
"""

In [36]:
groundness_checker = ChatOpenAI(
    model="gpt-4o",
    temperature=0
)

In [37]:
def evaluate(system_message,user_message_template, question, model=groundness_checker):
    retrieved_chunks=retriever.invoke(question)
    context_list = [d.page_content for d in retrieved_chunks]
    context = ". ".join(context_list)
    answer = RAG(question)
    prompt = f"""
     {system_message}\n
     USER:
     {user_message_template.format(question=question, context=context, answer=answer)}
    """
    juge_response= model.invoke(prompt)
    return juge_response.content


In [38]:
resp=evaluate(groundedness_rater_system_message, user_message_template, user_input)

In [39]:
print(display(Markdown(resp)))

**Étapes pour évaluer la réponse selon la métrique :**

1. **Identifier la question :** La question demande les titres et les notes des 10 meilleurs films.
   
2. **Analyser le contexte :** Le contexte fournit une liste de films classés par note IMDb, avec les titres et les notes correspondantes.

3. **Comparer la réponse avec le contexte :** Vérifier si les titres et les notes des films dans la réponse correspondent aux 10 premiers films listés dans le contexte.

4. **Vérifier l'exactitude des informations :** S'assurer que les titres et les notes sont correctement extraits du contexte.

5. **Évaluer la conformité à la métrique :** Déterminer si la réponse est entièrement dérivée des informations fournies dans le contexte.

**Explication étape par étape :**

1. **Identifier la question :** La question demande les titres et les notes des 10 meilleurs films selon le classement IMDb.

2. **Analyser le contexte :** Le contexte présente un classement des films avec leurs notes IMDb. Les 10 premiers films sont :
   - The Shawshank Redemption (1994) - 9.2
   - The Godfather (1972) - 9.2
   - The Godfather: Part II (1974) - 9.0
   - The Dark Knight (2008) - 8.9
   - Pulp Fiction (1994) - 8.9
   - The Good, the Bad and the Ugly (1966) - 8.9
   - Schindler's List (1993) - 8.9
   - 12 Angry Men (1957) - 8.9
   - The Lord of the Rings: The Return of the King (2003) - 8.9
   - Fight Club (1999) - 8.8

3. **Comparer la réponse avec le contexte :** La réponse de l'IA liste les mêmes films et notes que ceux trouvés dans le contexte pour les 10 premiers films.

4. **Vérifier l'exactitude des informations :** Les titres et les notes dans la réponse correspondent exactement à ceux du contexte.

5. **Évaluer la conformité à la métrique :** La réponse est entièrement dérivée des informations fournies dans le contexte, sans ajout ni omission.

**Évaluation :**

La réponse respecte entièrement la métrique, car elle est directement dérivée des informations fournies dans le contexte. Les titres et les notes des films correspondent exactement à ceux listés dans le contexte pour les 10 premiers films.

**Score : 5**

None


# **Conclusion**
- Dans ce notebook, nous avons appris à créer une application basée sur la génération augmentée par récupération (Retrieval-Augmented Generation, RAG), capable d’effectuer des questions-réponses sur des articles de recherche afin de récupérer des informations plus rapidement, plus efficacement et avec davantage de précision.